## Parte 5 · Unificación y conclusión (10 %)
1. Junta los cuatro orígenes en una única lista de diccionarios con las mismas claves y los mismos tipos, añadiendo un campo `origen` .

2. Calcula el importe total y el número de licitaciones por órgano.

3. Responde por escrito: si tuvieras que publicar este conjunto unificado para que lo consumieran otros equipos, ¿en cuál de los cuatro formatos lo harías y por qué? Argumenta con criterios técnicos, no por preferencia.

### 5.1. Unificación de los orígenes

Se normaliza cada registro a las mismas seis claves y tipos. El campo `origen` indica de qué fichero procede.

```python
def normaliza(d, origen):
    return {
        "id":          str(d["id"]),
        "organo":      str(d["organo"]),
        "objeto":      str(d["objeto"]),
        "importe":     float(d["importe"]),
        "fechaLimite": d["fechaLimite"],   # ya es datetime en partes 1, 2 y 4
        "origen":      origen,
    }

unificados = (
    [normaliza(x, "json") for x in validos_json] +
    [normaliza(x, "xml")  for x in validos_xml]  +
    [normaliza(x, "html") for x in validos_html]
)

### 5.2. Aplicación del umbral del YAML

El umbral `IMPORTE_MIN` se lee de `analisis.yaml` (parte 3) y se aplica a la lista unificada. No hay ningún valor escrito a mano.

```python
IMPORTE_MIN = config["validacion"]["importe_minimo"]   # 100000.0

unificados_filtrados = [
    x for x in unificados if x["importe"] >= IMPORTE_MIN
]

### 5.4. ¿En qué formato publicar el conjunto unificado?

**Recomendación: JSON.**

#### Criterios técnicos

| Criterio | JSON | XML | YAML | HTML |
|---|---|---|---|---|
| Modelo nativo para `list[dict]` | Sí | No: hay que inventar jerarquía | Sí, pero débil | No: presentación |
| Parsing en cualquier lenguaje | Directo | Requiere namespaces | Requiere `safe_load` | Parser tolerante |
| Validación con esquema | JSON Schema | XSD | Sin estándar equivalente | No aplica |
| Tipos de datos | Número, cadena, booleano, nulo | Vía XSD | Débil (`no`→`False`, `1.10`→`1.1`) | Todo texto |
| Compacto y legible | Sí | Verboso | Muy compacto pero frágil | No es datos |

#### Argumentos

1. El conjunto es una **lista de registros homogéneos** (mismas claves, tipos, campo `origen`). JSON lo representa tal cual; XML obliga a diseñar etiquetas y namespaces que no aportan.
2. La fecha se serializa como **ISO 8601** (`datetime.isoformat()`), igual que en las partes 1 y 2. Con JSON Schema se refuerza con `format: date-time`.
3. **JSON Schema** fija el contrato: campos obligatorios, tipos (`number` para importe) y `enum` para `origen`. Los consumidores validan antes de procesar.
4. **YAML no es formato de intercambio**: es configuración. Depende de indentación, admite interpretaciones ambiguas y `load()` es riesgo de seguridad.
5. **HTML es presentación**: sirve como vista previa, no como fuente de datos.
6. **XML solo si el consumidor lo exige** (sistemas CODICE/Atom). Se ofrece como formato secundario; la fuente de verdad es JSON.

#### Formato propuesto

```json
{
  "version": "1.0",
  "generado": "2026-11-18T10:00:00",
  "licitaciones": [
    {
      "id": "2026-API-0001",
      "organo": "Ayuntamiento de Valencia",
      "objeto": "Suministro de servidores de cálculo…",
      "importe": 184500.0,
      "fechaLimite": "2026-10-15T14:00:00",
      "origen": "json"
    }
  ]
}